# UHCL: Unified Hierarchical Contrastive Loss

This is a PyTorch implementation of **UHCL**, a loss function that unifies representation learning, anomaly detection, and hierarchical contrastive learning.

## Data-Space Hierarchy

UHCL operates on a hierarchical partitioning of the data-space:

```
Noise (random patterns)
  ↓
Outliers (meaningful but OOD)
  ↓
Inliers (target distribution)
  ↓
Positive Inliers (same semantic class)
```

The framework maintains similarity constraints across these levels, ensuring:
- High similarity within positive inlier pairs
- Separation between inliers and outliers
- Separation between meaningful patterns and noise


## Quick Start

```python
# 1. Setup model and data
model = SmallResNet(embed_dim=128, pretrained=True, num_classes=3)
model = model.to(device)

# 2. Sample hierarchical batch
X_batch, yfine, ycoarse = sample_hierarchy_batch(
    in_dataset, out_dataset, noise_dataset,
    batch_size=100, device=device, N=6, is_hierarchical=True
)

# 3. Forward pass and loss computation
B = X_batch.shape[0]
X_flat = X_batch.view(B * 6, 3, 32, 32)  # Flatten batch×hierarchy
emb_flat, z_flat = model(X_flat)
emb = emb_flat.view(B, 6, -1)  # [B, 6, 128]

loss = uhcl(emb, N=6, alpha=0.95, device=device)

# 4. Backward
optimizer.zero_grad()
loss.backward()
optimizer.step()
```


# 1. IMPORTS & CONFIGURATION

## Configuration Dictionary

Each experiment parameter is centralized in a `config` dict for easy reproducibility and hyperparameter tuning.

| Parameter | Role | Typical Value |
|-----------|------|-------|
| `alpha` | Margin scale in contrastive loss | 0.8 - 0.95 |
| `alpha_decay` | Factor for margin decay per level | 0.8 - 0.9 |
| `embed_dim` | Dimension of L2-normalized embeddings | 128 |
| `lr` | Learning rate for Adam optimizer | 4e-5 |
| `anchors_per_fine` | Number of anchors per fine class | 2 |
| `pretrained` | Use ImageNet pre-trained backbone | True |
| `batch_size` | Batch size | 100 |
| `epochs` | Training epochs | 100 |


In [ ]:
import os
import random
from typing import List, Tuple, Dict
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
from sklearn.metrics import roc_auc_score
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from collections import defaultdict

# CONFIG - Hyperparameters


In [ ]:
config = {
    "experiment_name": "2222", # Changed experiment name to avoid overwriting
    "inlier_dataset": "cifar100",
    "outlier_dataset": "cifar100",
    
    "epochs":100,
    "batch_size": 100,
    "anchors_per_fine": 2,
    "lr": 4e-5,
    "aux_weight": 0.000,
    "weight_decay": 0.000,   
    "alpha": 0.01,              # alpha[0] = alpha
    "alpha_decay": 0.9,         # alpha[j] = alpha * (alpha_decay ** j)
                                # the hierarchical design of UHCL ensures alpha[j] in each hiearchy and 
                                # it doesn't require calculation inside hiearchical loop
    "lambda_pos": 0.0,
    "embed_dim": 2,
    "num_workers": 8,
    "pretrained": True,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "ckpt_dir": "./content/checkpoints",
    "resume_ckpt_path": None, # If specified uses a .pth file to resume trainig
                              # if None it does the training from scracth (pretrained or random initilized)
}
os.makedirs(config["ckpt_dir"], exist_ok=True)

def set_seed(seed=39):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

class ToTensorSafe:
    def __call__(self, pic):
        if isinstance(pic, torch.Tensor):
            return pic.float() / 255. if pic.max() > 1 else pic.float()
        return transforms.functional.to_tensor(pic)

# 2. DATA LOADERS & TRANSFORMS

## Transform Strategy

Different augmentation pipelines for grayscale vs. color datasets:

**Grayscale Datasets** (MNIST, FashionMNIST, EMNIST):
- Minimal augmentation (just normalization)
- Reasoning: Limited color information to augment

**Color Datasets** (CIFAR-10, CIFAR-100):
- ColorJitter, RandomGrayscale, ImageNet normalization
- Helps learn robust representations invariant to color changes


In [ ]:
IMG_SIZE = {"mnist": 28, "fashionmnist": 28, "emnist": 28, "cifar10": 32, "cifar100": 32, "svhn": 32}

def get_transforms(name, mode="train"):
    size = IMG_SIZE.get(name, 32)
    tensorizer = ToTensorSafe()
    if name in ["mnist", "fashionmnist", "emnist"]:
        # grayscale datasets: no ImageNet norm
        if mode == "train":
            return transforms.Compose([tensorizer])
        else:
            return transforms.Compose([transforms.Resize(size), tensorizer])

    # color image datasets (CIFAR*)
    if mode == "train":
        return transforms.Compose([
            # Color augmentations
            transforms.ColorJitter(
                brightness=0.4,
                contrast=0.4,
                saturation=0.4,
                hue=0.1
            ),
            
            transforms.RandomGrayscale(p=0.2),    # 20% chance grayscale
            
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)
            ),
        ])
    
    else:
        # Test-time: no augmentation except resizing + normalization
        return transforms.Compose([
            transforms.Resize(size),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)
            ),
        ])

class RandomNoiseDataset(Dataset):
    """Generates random uniform noise for the noise hierarchy level."""
    def __init__(self, num: int, shape: Tuple[int, int, int]):
        self.num = num; self.shape = shape
    def __len__(self): return self.num
    def __getitem__(self, idx): return torch.rand(self.shape), -1

## Custom Dataset Classes

### CIFAR100Hierarchy
Extends torchvision's CIFAR100 to expose both fine (100 classes) and coarse (20 superclasses) labels.

**Key Features**:
- Pre-computed `fine_label_indices` & `coarse_label_indices` for efficient sampling
- `__getitem__()` returns `(image, fine_label, coarse_label)`
- Enables N-way contrastive sampling: anchor → same fine → same coarse → diff coarse

### CIFAR10Indexed
Flat wrapper with pre-computed label indices for faster sampling.


In [ ]:
import copy
import random
import numpy as np
import torch
from torch.utils.data import Subset

def get_random_coarse_split(dataset, split_ratio=0.5, seed=None):
    """
    Randomly selects a subset of coarse labels to be 'inliers'.
    Returns a set of inlier coarse labels.
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    all_coarse_labels = list(dataset.coarse_label_indices.keys())
    random.shuffle(all_coarse_labels)
    
    n_inliers = int(len(all_coarse_labels) * split_ratio)
    inlier_labels = set(all_coarse_labels[:n_inliers])
    
    print(f"Split generated: {len(inlier_labels)} Inlier Coarse Classes")
    return inlier_labels

def split_dataset_by_labels(full_dataset, inlier_coarse_labels):
    """
    Splits a dataset into in/out sets based on a PRE-DEFINED set of coarse labels.
    """
    # 1. Identify Outlier Indices for the 'out_dataset'
    #    Iterate through all coarse labels in the dataset; if not in inlier list, it's an outlier.
    outlier_indices = []
    dataset_coarse_keys = full_dataset.coarse_label_indices.keys()
    
    for c_lbl in dataset_coarse_keys:
        if c_lbl not in inlier_coarse_labels:
            outlier_indices.extend(full_dataset.coarse_label_indices[c_lbl])
            
    out_dataset = Subset(full_dataset, outlier_indices)

    # 2. Prepare Inlier Dataset (Deep copy indices to avoid mutation issues)
    in_dataset = copy.copy(full_dataset)
    in_dataset.fine_label_indices = copy.deepcopy(full_dataset.fine_label_indices)
    in_dataset.coarse_label_indices = copy.deepcopy(full_dataset.coarse_label_indices)

    # Filter Coarse Indices: Keep only inlier keys
    in_dataset.coarse_label_indices = {
        c: idxs for c, idxs in in_dataset.coarse_label_indices.items() 
        if c in inlier_coarse_labels
    }

    # Filter Fine Indices: Keep only fine labels belonging to inlier coarse classes
    valid_fine_labels = []
    for fine_lbl, indices in in_dataset.fine_label_indices.items():
        if not indices: continue
        
        # Check coarse label of the first sample in this fine class
        _, _, sample_coarse = full_dataset[indices[0]]
        
        if sample_coarse in inlier_coarse_labels:
            valid_fine_labels.append(fine_lbl)
            
    in_dataset.fine_label_indices = {
        f: in_dataset.fine_label_indices[f] for f in valid_fine_labels
    }

    return in_dataset, out_dataset

In [ ]:
import torch
from torchvision import datasets
import numpy as np
from collections import defaultdict

class CIFAR100Hierarchy(datasets.CIFAR100):
    """
    Custom CIFAR-100 dataset that exposes both fine and coarse labels.
    CIFAR-100 has 20 coarse categories (superclasses), each containing 5 fine categories.
    Includes indexing for efficient sampling.
    """
    def __init__(self, root, train=True, transform=None, target_transform=None, download=False):
        super().__init__(root, train=train, transform=transform,
                         target_transform=target_transform, download=download)

        # Define the hierarchy: mapping from fine labels to coarse labels
        self.coarse_to_fine = {
            0: [4, 30, 55, 72, 95],   # aquatic mammals
            1: [1, 32, 67, 73, 91],   # fish
            2: [54, 62, 70, 82, 92],  # flowers
            3: [9, 10, 16, 28, 61],   # food containers
            4: [0, 51, 53, 57, 83],   # fruit and vegetables
            5: [22, 39, 40, 86, 87],  # household electrical devices
            6: [5, 20, 25, 84, 94],   # household furniture
            7: [6, 7, 14, 18, 24],    # insects
            8: [3, 42, 43, 88, 97],   # large carnivores
            9: [12, 17, 37, 68, 76],  # large man-made outdoor things
            10: [23, 33, 49, 60, 71], # large natural outdoor scenes
            11: [15, 19, 21, 31, 38], # large omnivores and herbivores
            12: [34, 63, 64, 66, 75], # medium-sized mammals
            13: [26, 45, 77, 79, 99], # non-insect invertebrates
            14: [2, 11, 35, 46, 98],  # people
            15: [27, 29, 44, 78, 93], # reptiles
            16: [36, 50, 65, 74, 80], # small mammals
            17: [47, 52, 56, 59, 96], # trees
            18: [8, 13, 48, 58, 90],  # vehicles 1
            19: [41, 69, 81, 85, 89], # vehicles 2
        }

        # Create a mapping from fine labels to coarse labels
        self.fine_to_coarse = {}
        for coarse_label, fine_labels in self.coarse_to_fine.items():
            for fine_label in fine_labels:
                self.fine_to_coarse[fine_label] = coarse_label

        # Create coarse_targets array for all samples
        self.coarse_targets = [self.fine_to_coarse[fine_label] for fine_label in self.targets]

        # Create indices for efficient sampling
        self.fine_label_indices = defaultdict(list)
        self.coarse_label_indices = defaultdict(list)

        for i, (fine_label, coarse_label) in enumerate(zip(self.targets, self.coarse_targets)):
            self.fine_label_indices[fine_label].append(i)
            self.coarse_label_indices[coarse_label].append(i)

    def __getitem__(self, index):
        img, fine_target = super().__getitem__(index)
        coarse_target = self.coarse_targets[index]
        return img, fine_target, coarse_target

class CIFAR10Indexed(datasets.CIFAR10):
    """Custom CIFAR-10 dataset with pre-computed indices for faster sampling."""
    def __init__(self, root, train=True, transform=None, target_transform=None, download=False):
        super().__init__(root, train=train, transform=transform,
                         target_transform=target_transform, download=download)
        self.label_indices = defaultdict(list)
        for i, label in enumerate(self.targets):
            self.label_indices[label].append(i)

# 3. HIERARCHICAL BATCH SAMPLING

## Purpose

Create training tuples that respect the data-space hierarchy:

```
Inlier Hierarchy:
  x0: Anchor (any fine class)
  x1: Positive (same fine class as x0)
  x2: Negative inlier (same coarse as x0, different fine)
  x3: Negative inlier (different coarse from x0)
  x4: Outlier (out-of-distribution)
  x5: Noise (random pattern)
```

## Sampling Strategy

For each fine class:
1. Sample `anchors_per_fine` anchor samples
2. For each anchor:
   - x1: Random sample from same fine class
   - x2: Random sample from same coarse, different fine
   - x3: Random sample from different coarse
   - x4: Random outlier
   - x5: Random noise
3. Stack into tuple [6, 3, 32, 32]


In [ ]:
def sample_hierarchy_batch(in_dataset, out_dataset, noise_dataset,
                           batch_size, device, N = 4, is_hierarchical=False,
                           anchors_per_fine=config["anchors_per_fine"]):

    X_batch = []
    yfine_batch = []
    ycoarse_batch = []
    y_batch = []

    if is_hierarchical:
        fine_labels = list(in_dataset.fine_label_indices.keys())
        random.shuffle(fine_labels)

        for fine_label in fine_labels:
            fine_indices = in_dataset.fine_label_indices[fine_label]
            if len(fine_indices) < anchors_per_fine:
                anchors = np.random.choice(fine_indices, anchors_per_fine, replace=True)
            else:
                anchors = np.random.choice(fine_indices, anchors_per_fine, replace=False)

            for idx in anchors:
                try:
                    x0, fine0, coarse0 = in_dataset[idx]
                    pos_candidates = [i for i in in_dataset.fine_label_indices[fine0] if i != idx]
                    pos_idx = random.choice(pos_candidates)
                    x1, _, _ = in_dataset[pos_idx]

                    coarse_candidates = [
                        i for i in in_dataset.coarse_label_indices[coarse0]
                        if in_dataset.targets[i] != fine0
                    ]
                    neg_fine_idx = random.choice(coarse_candidates)
                    x2, fine2, _ = in_dataset[neg_fine_idx]

                    all_coarse = list(in_dataset.coarse_label_indices.keys())
                    diff_coarse = [c for c in all_coarse if c != coarse0]
                    rand_coarse = random.choice(diff_coarse)
                    diff_coarse_candidates = in_dataset.coarse_label_indices[rand_coarse]
                    diff_coarse_idx = random.choice(diff_coarse_candidates)
                    x3, fine3, coarse3 = in_dataset[diff_coarse_idx]

                    x4, *rest = out_dataset[np.random.randint(len(out_dataset))]
                    x5, _ = noise_dataset[np.random.randint(len(noise_dataset))]

                    X_tuple = torch.stack([x0, x1, x2, x3, x4, x5])
                    yfine_tuple = [fine0, fine0, fine2, fine3]
                    ycoarse_tuple = [coarse0, coarse0, coarse0, coarse3]

                    X_batch.append(X_tuple)
                    yfine_batch.append(yfine_tuple)
                    ycoarse_batch.append(ycoarse_tuple)

                except Exception as e:
                    print(f'Sampling error: {e}')
                    x0, _, _ = in_dataset[idx]
                    fallback = [x0] * N
                    X_batch.append(torch.stack(fallback))

    else:
        # CIFAR10-like (flat, no coarse labels)
        labels = list(in_dataset.label_indices.keys())
        random.shuffle(labels)
        for label in labels:
            label_indices = in_dataset.label_indices[label]
            if len(label_indices) < anchors_per_fine:
                anchors = np.random.choice(label_indices, anchors_per_fine * 10, replace=True)
            else:
                anchors = np.random.choice(label_indices, anchors_per_fine * 10, replace=False)
            for idx in anchors:
                try:
                    x0, y0 = in_dataset[idx]
                    pos_candidates = [i for i in in_dataset.label_indices[y0] if i != idx]
                    pos_idx = random.choice(pos_candidates) if pos_candidates else idx
                    x1, _ = in_dataset[pos_idx]
                    diff_labels = [l for l in labels if l != y0]
                    neg_label = random.choice(diff_labels)
                    neg_idx = random.choice(in_dataset.label_indices[neg_label])
                    x2, y2 = in_dataset[neg_idx]
                    x3, _ = out_dataset[np.random.randint(len(out_dataset))]
                    x4, _ = noise_dataset[np.random.randint(len(noise_dataset))]
                    X_tuple = torch.stack([x0, x1, x2, x3, x4])
                    y_tuple = [y0, y0, y2]
                    X_batch.append(X_tuple)
                    y_batch.append(y_tuple)
                except Exception as e:
                    print(f'Sampling error: {e}')
                    x0, _ = in_dataset[idx]
                    fallback = [x0] * N
                    X_batch.append(torch.stack(fallback))
        yfine_batch = y_batch
        ycoarse_batch = None

    if len(X_batch) > batch_size:
        X_batch = X_batch[:batch_size]
        if yfine_batch is not None:
            yfine_batch = yfine_batch[:batch_size]
        if ycoarse_batch is not None:
            ycoarse_batch = ycoarse_batch[:batch_size]

    X_batch = torch.stack(X_batch).to(device)
    if ycoarse_batch is None:
        ycoarse_batch = torch.zeros((len(yfine_batch), 1), dtype=torch.long)

    return (
        X_batch,
        torch.tensor(yfine_batch, dtype=torch.long, device=device),
        torch.tensor(ycoarse_batch, dtype=torch.long, device=device),
    )

# 4. MODEL ARCHITECTURE

## SmallResNet Architecture

```
Input [B, C, H, W]
  ↓
ResNet18/50 Backbone (removes final classification layer)
  ↓
AdaptiveAvgPool2d(1, 1) → [B, 512, 1, 1]
  ↓
Flatten() → [B, 512]
  ↓
Linear(512, 512) → ReLU → Linear(512, embed_dim)
  ↓
L2 Normalization → [B, embed_dim] (on unit hypersphere)
```

**Key Design Choices**:
- **L2 Normalization**: Maps embeddings to unit hypersphere (critical for contrastive learning)
- **AdaptiveAvgPool2d**: Ensures 1×1 spatial dims regardless of input resolution
- **Two-layer MLP**: Projects through 512-dim intermediate for non-linearity
- **Optional clf_head**: For downstream classification (inlier/outlier/noise)


In [ ]:
class SmallResNet(nn.Module):
    def __init__(self, embed_dim=128, pretrained=False, in_channels=3, depth=18, num_classes=None):
        super().__init__()
        if depth == 50:
            net = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
            proj_in_features = 2048
        else:
            net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
            proj_in_features = 512

        if in_channels != 3:
            original_conv1 = net.conv1
            net.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
            if pretrained:
                net.conv1.weight.data[:, :3, :, :] = original_conv1.weight.data
                nn.init.kaiming_normal_(net.conv1.weight.data[:, 3:, :, :], mode='fan_out', nonlinearity='relu')

        self.backbone = nn.Sequential(*list(net.children())[:-1])
        self.projector = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(proj_in_features, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, embed_dim),
        )

        self.clf_head = None
        if num_classes is not None:
            self.clf_head = nn.Linear(embed_dim, num_classes)

    def forward(self, x, return_embeddings=True):
        z = self.backbone(x)
        z = self.projector(z)
        emb = F.normalize(z, p=2, dim=1)  # L2 normalization
        return emb, z

    def forward_features_and_logits(self, x):
        emb = self(x)
        if self.clf_head is None:
            raise ValueError("clf_head is not defined. Make sure num_classes was set.")
        logits = self.clf_head(emb)
        return emb, logits

# 5. UHCL LOSS FUNCTION

## Mathematical Formulation

### Step 1: Similarity Matrix
Given normalized embeddings [B, N, D] (batch, hierarchy levels, dimension):
```
S = emb @ emb^T  # [B, N, N]
S[b, i, j] = cos_similarity(emb[b, i], emb[b, j])
```

### Step 2: Margin Constraints
For each hierarchy level j and preceding level i:
```
for k in [i, j]:  # All negatives at next level
    loss += max(0, S[k, j+1] - S[i, j] + α)
```

This ensures positive similarity >> negative similarity + margin.

### Step 3: Normalization
Normalize by:
- λ_ij = 1/(j - i + 1)  # Number of negative pairs
- λ_j = 1/j              # Number of positive levels

### Loss Equation
$$\mathcal{L} = \sum_{j=1}^{N-1} \sum_{i=0}^{j-1} \lambda_{ij} \sum_{k=i}^{j} \max(S_{kj+1} - S_{ij} + \alpha_j, 0)$$


In [ ]:
def uhcl(emb, N=5, alpha=0.95, alpha_decay=0.8, device=None):
    """
    Unified Hierarchical Contrastive Loss
    
    Args:
        emb: [B, N, D] normalized embeddings (batch_size, n_levels, embed_dim)
        N: number of hierarchy levels
        alpha: margin scale
        alpha_decay: margin decay factor
        device: torch device
    
    Returns:
        Scalar loss tensor
    """
    B, total_levels, D = emb.shape
    S = torch.bmm(emb, emb.transpose(1, 2))  # [B, N, N] similarity matrix

    Total_loss = torch.tensor(0.0, device=device)

    # Margin loss: anchor vs. final level
    total_margin_loss = torch.tensor(0.0, device=device)
    total_margin_loss += F.relu(S[:,0,total_levels-1]).mean()
    
    # Adaptive margins based on hierarchy depth
    if total_levels == 6:
        margin = [15/16, 7/8, 3/4, 1/2]  # Decreasing margins
    else:
        margin = [7/8, 3/4, 1/2]

    # Margin enforcement
    for j in range(1, total_levels - 2):
        margin_loss = F.relu(margin[j-1] - S[:,0,j])
        total_margin_loss += margin_loss.mean()
    
    # Hierarchical hinge losses
    for j in range(1, total_levels - 1):
        alpha_j = (1 - margin[j-1]) * 0.95  # Dynamic margin per level
        hierarchy_j_loss = torch.tensor(0.0, device=device)

        for i in range(0, j):
            partial_loss = torch.tensor(0.0, device=device)

            # Hinge loss for all negative pairs
            for k in range(i, j + 1):
                positive_sim = S[:, i, j]
                negative_sim = S[:, k, j + 1]
                hinge_loss = F.relu(negative_sim - positive_sim + alpha_j)
                partial_loss += hinge_loss.mean()

            # Normalize by number of negative pairs
            lambda_ij = 1 / (j - i + 1)
            hierarchy_j_loss += partial_loss * lambda_ij

        # Normalize by number of positive levels
        lambda_j = 1 / j
        Total_loss += hierarchy_j_loss * lambda_j

    return Total_loss + total_margin_loss

# 6. EVALUATION

## Linear Probe Training (`train_clf`)

Train a lightweight classification head on frozen backbone features. This evaluates the quality of learned representations.

**Process**:
1. Freeze backbone (no gradient updates)
2. Extract features from all training data
3. Train linear layer to classify inlier/outlier/noise
4. Report accuracy on validation data

## Feature Extraction (`extract_embeddings`)

Extract learned representations for downstream evaluation (k-NN, t-SNE, etc.)

**Returns**:
- all_features: [N, D] embedding vectors
- all_fine_labels: [N] class labels
- all_coarse_labels: [N] superclass labels (if hierarchical)
- all_classifier_probs: [N, num_classes] classification probabilities

## K-NN Evaluation (`compute_knn`)

Train k-NN classifier on embeddings using cosine distance metric.
Reports accuracy, precision, recall, F1-score.


In [ ]:
def train_clf(model, in_train, out_train, noise_train, device, epochs=10, batch_size=256, lr=1e-3):
    """
    Trains the 3-class classification head (Inlier=0, Outlier=1, Noise=2)
    using frozen embeddings from the trained backbone.
    """
    print("\n--- Training 3-Class Classification Head (Linear Probe) ---")
    
    model.eval()  # Freeze backbone
    if model.clf_head is None:
        print("Error: Classification head not initialized.")
        return

    @torch.no_grad()
    def get_feats_labels(dataset, label_val):
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4)
        feats = []
        targets = []
        for batch in tqdm(loader, desc=f"Class {label_val}"):
            if isinstance(batch, (list, tuple)):
                x = batch[0]
            else:
                x = batch
            x = x.to(device)
            _, logits = model(x)
            feats.append(logits.cpu())
            targets.append(torch.full((logits.size(0),), label_val))
        return torch.cat(feats), torch.cat(targets)

    # Extract features for all 3 categories
    in_feats, in_labels = get_feats_labels(in_train, 0)
    out_feats, out_labels = get_feats_labels(out_train, 1)
    noise_feats, noise_labels = get_feats_labels(noise_train, 2)

    X_all = torch.cat([in_feats, out_feats, noise_feats])
    y_all = torch.cat([in_labels, out_labels, noise_labels])

    head_dataset = torch.utils.data.TensorDataset(X_all, y_all)
    head_loader = DataLoader(head_dataset, batch_size=batch_size, shuffle=True)

    head_opt = torch.optim.Adam(model.clf_head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    model.clf_head.train()
    for ep in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        for feats, labels in head_loader:
            feats, labels = feats.to(device), labels.to(device)
            logits = model.clf_head(feats)
            loss = criterion(logits, labels.long())
            head_opt.zero_grad()
            loss.backward()
            head_opt.step()
            total_loss += loss.item()
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        avg_loss = total_loss / len(head_loader)
        acc = 100 * correct / total
        print(f"Head Epoch {ep+1}/{epochs} | Loss: {avg_loss:.4f} | Acc: {acc:.2f}%")

    print("Classification Head Training Complete.")

In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

@torch.no_grad()
def extract_embeddings(dataset, model, device, batch_size=256, hierarchical=False):
    """
    Extract learned representations for evaluation.
    
    Returns:
        all_features: [N, D] embedding vectors
        all_fine_labels: [N] class labels
        all_coarse_labels: [N] superclass labels
        all_classifier_probs: [N, num_classes] classification probabilities
    """
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_features = []
    all_fine_labels = []
    all_coarse_labels = []
    all_classifier_probs = []

    for batch in loader:
        if hierarchical:
            images, fine_labels, coarse_labels = batch
        else:
            images, fine_labels = batch
            coarse_labels = [-1] * len(fine_labels)

        images = images.to(device)
        _, features_t = model(images)
        features_np = features_t.detach().cpu().numpy()
        all_features.append(features_np)

        all_fine_labels.extend([int(l) for l in fine_labels])
        all_coarse_labels.extend([int(l) for l in coarse_labels])

        if getattr(model, "clf_head", None) is not None:
            features_tensor = torch.tensor(features_np, dtype=torch.float32).to(device)
            logits = model.clf_head(features_tensor)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            all_classifier_probs.append(probs)

    all_features = np.vstack(all_features)
    all_fine_labels = np.array(all_fine_labels)
    all_coarse_labels = np.array(all_coarse_labels)
    if len(all_classifier_probs):
        all_classifier_probs = np.vstack(all_classifier_probs)
    else:
        all_classifier_probs = np.empty((len(all_features), 0))

    return all_features, all_fine_labels, all_coarse_labels, all_classifier_probs

def compute_knn(train_features, train_labels, test_features, test_labels, k=20):
    """Train k-NN classifier with cosine distance metric."""
    knn = KNeighborsClassifier(n_neighbors=k, metric='cosine')
    knn.fit(train_features, train_labels)

    predictions = knn.predict(test_features)
    accuracy = (predictions == test_labels).mean()
    precision = precision_score(test_labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(test_labels, predictions, average='weighted', zero_division=0)
    f1 = f1_score(test_labels, predictions, average='weighted', zero_division=0)

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'predictions': predictions
    }

# 7. TRAINING LOOP TEMPLATE

## Complete End-to-End Training Example

```python
# 1. Setup
model = SmallResNet(embed_dim=128, pretrained=True, num_classes=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=4e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])

# 2. Load data
in_dataset = CIFAR100Hierarchy('./data', train=True, transform=train_transform)
out_dataset = RandomNoiseDataset(50000, (3, 32, 32))
noise_dataset = RandomNoiseDataset(50000, (3, 32, 32))

# 3. Training loop
for epoch in range(config['epochs']):
    # Sample hierarchical batch
    X_batch, yfine, ycoarse = sample_hierarchy_batch(
        in_dataset, out_dataset, noise_dataset,
        batch_size=config['batch_size'],
        device=device,
        N=6,
        is_hierarchical=True
    )
    
    # Forward pass
    B = X_batch.shape[0]
    X_flat = X_batch.view(B * 6, 3, 32, 32)  # Flatten batch×hierarchy
    emb_flat, z_flat = model(X_flat)
    emb = emb_flat.view(B, 6, -1)  # Reshape [B, 6, 128]
    
    # Compute loss
    loss = uhcl(emb, N=6, alpha=config['alpha'], device=device)
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()
    
    # Periodic evaluation
    if (epoch + 1) % 10 == 0:
        features, fine_labels, coarse_labels, probs = extract_embeddings(
            test_dataset, model, device, hierarchical=True
        )
        # Evaluate metrics...
```


# 8. TROUBLESHOOTING

## Common Issues & Solutions

### Issue 1: "RuntimeError: expected 3D input (got 2D input)" in UHCL
**Cause**: Embeddings not reshaped to [B, N, D] before UHCL loss.

**Fix**:
```python
# WRONG:
emb_flat = model(X_batch)  # [B*6, 128]
loss = uhcl(emb_flat, ...)

# CORRECT:
B = X_batch.shape[0]
emb_flat = model(X_batch)  # [B*6, 128]
emb = emb_flat.view(B, 6, 128)  # Reshape to [B, 6, 128]
loss = uhcl(emb, ...)
```

### Issue 2: Slow batch sampling
**Cause**: `sample_hierarchy_batch()` loops over all classes sequentially.

**Solution**: Use vectorized operations or cache label indices.

### Issue 3: Poor outlier detection on unseen OOD
**Cause**: Limited outlier diversity during training.

**Solution**:
- Use multiple diverse outlier sources
- Implement hard negative mining
- Increase outlier sampling frequency
